# First look at the data

Exploration only, not pipeline code. Goals:
1. Download 5 years of **raw** AAPL prices (`auto_adjust=False`) and look at their shape.
2. Look at AAPL's splits and dividends.
3. Save the S&P 500 ticker list to `data/reference/sp500_tickers.csv`.

In [ ]:
from io import StringIO
from pathlib import Path

import pandas as pd
import requests
import yfinance as yf

print("yfinance", yf.__version__, "| pandas", pd.__version__)

## 1. Daily prices: AAPL, last 5 years

In [ ]:
# auto_adjust=False keeps raw Close and gives a separate 'Adj Close' column.
prices = yf.download("AAPL", period="5y", auto_adjust=False, progress=False)
print("shape:", prices.shape)
print("index:", prices.index.dtype, "| from", prices.index.min().date(), "to", prices.index.max().date())
print("columns:", list(prices.columns))
prices.tail()

In [ ]:
# Column header is a 2-level MultiIndex (Price, Ticker), even for one ticker.
# Flatten it so the frame is easier to read.
aapl = prices.xs("AAPL", axis=1, level="Ticker")
aapl.dtypes

In [ ]:
# Basic quality checks we'll later write as dbt tests.
print("rows:", len(aapl))
print("nulls per column:", aapl.isna().sum().to_dict())
print("duplicate dates:", aapl.index.duplicated().sum())
print("high >= low everywhere:", (aapl["High"] >= aapl["Low"]).all())
print("all prices > 0:", (aapl[["Open", "High", "Low", "Close", "Adj Close"]] > 0).all().all())
aapl.describe().round(2)

In [ ]:
# Close vs Adj Close: how big is the dividend adjustment?
ratio = (aapl["Adj Close"] / aapl["Close"]).rename("adj_ratio")
pd.concat([ratio.head(3), ratio.tail(3)])

## 2. Splits and dividends

In [ ]:
ticker = yf.Ticker("AAPL")
splits = ticker.splits
print("index dtype:", splits.index.dtype)
splits

In [ ]:
dividends = ticker.dividends
print("dividend rows (full history):", len(dividends))
dividends.tail(8)

### Is `Close` really raw? Check the 2020-08-31 4:1 split

The 5-year window starts after AAPL's last split, so it doesn't show what a split does to the prices.
Pull the days around the August 2020 split. On the real tape, AAPL closed at about **$499** on 2020-08-28 and about **$129** on 2020-08-31.

In [ ]:
around_split = yf.download("AAPL", start="2020-08-25", end="2020-09-03", auto_adjust=False, progress=False)
around_split.xs("AAPL", axis=1, level="Ticker")[["Close", "Adj Close", "Volume"]]

## 3. S&P 500 ticker list from Wikipedia

In [ ]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
# Wikipedia returns 403 to Python's default User-Agent, so send our own.
headers = {"User-Agent": "stock-market-elt-pipeline/0.1 (portfolio project)"}
resp = requests.get(url, headers=headers, timeout=30)
resp.raise_for_status()

# Table 0 = current constituents. Read CIK as a string so the leading zeros
# survive (SEC EDGAR uses 10-digit zero-padded CIKs).
sp500 = pd.read_html(StringIO(resp.text), converters={"CIK": str})[0]
print("rows:", len(sp500), "| unique symbols:", sp500["Symbol"].nunique())
print("columns:", list(sp500.columns))
sp500.head()

In [ ]:
# Symbols with a dot: Yahoo uses a dash instead (BRK.B -> BRK-B).
sp500[sp500["Symbol"].str.contains(r"\.")][["Symbol", "Security"]]

In [ ]:
out = Path("../data/reference/sp500_tickers.csv")
sp500.to_csv(out, index=False)
print("saved", len(sp500), "rows to", out.resolve())